In [28]:
from getpass import getpass

In [29]:
password = getpass("Enter PostgreSQL password: ")

Enter PostgreSQL password: ··········


In [30]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))
df.head()

Saving customer_revenue_retention_analysis.csv to customer_revenue_retention_analysis (1).csv


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [31]:
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [32]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [33]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [34]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [35]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [36]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [37]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [38]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [39]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [40]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [41]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [42]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [43]:
df = df.drop('promo_code_used', axis=1)

In [44]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [45]:
!pip install psycopg2-binary sqlalchemy

In [46]:
!pip install -q sqlalchemy psycopg2-binary

In [47]:
from sqlalchemy import URL, create_engine

db_url = URL.create(
    "postgresql+psycopg2",
    username="neondb_owner",
    password=password,
    host="ep-cool-paper-aykqgijg-pooler.c-5.us-east-2.aws.neon.tech",
    port=5432,
    database="neondb",
    query={
        "sslmode": "require",
        "channel_binding": "require"
    }
)

engine = create_engine(db_url)

print("Connection created successfully!")

Connection created successfully!


In [48]:
with engine.connect() as conn:
    print("PostgreSQL connected successfully!")

PostgreSQL connected successfully!


In [49]:
with engine.connect() as conn:
    print("Neon PostgreSQL connected successfully!")

df.to_sql(
    "customer",
    engine,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into table 'customer'.")

Neon PostgreSQL connected successfully!
Data successfully loaded into table 'customer'.


In [50]:
!pip install pymysql sqlalchemy

In [51]:
import pandas as pd

with engine.connect() as conn:
    print("Neon PostgreSQL connected successfully! ")

table_name = "customer"

df.to_sql(
    table_name,
    engine,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into PostgreSQL!")

sample = pd.read_sql(
    "SELECT * FROM customer LIMIT 5;",
    engine
)

print(sample)

Neon PostgreSQL connected successfully! 
Data successfully loaded into PostgreSQL!
   customer_id  age gender item_purchased  category  purchase_amount  \
0            1   55   Male         Blouse  Clothing               53   
1            2   19   Male        Sweater  Clothing               64   
2            3   50   Male          Jeans  Clothing               73   
3            4   21   Male        Sandals  Footwear               90   
4            5   45   Male         Blouse  Clothing               49   

        location size      color  season  review_rating subscription_status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   shippi

**Advanced Customer & Revenue Analysis**

In [52]:
df['spending_band'] = pd.qcut(
    df['purchase_amount'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

In [53]:
df['spending_band'].value_counts()

,count
spending_band,
Low,1014
High,988
Medium,972
Very High,926


In [54]:
purchase_threshold = df['previous_purchases'].median()
spend_threshold = df['purchase_amount'].median()

df['customer_value_segment'] = 'Regular'

df.loc[
    (df['previous_purchases'] > purchase_threshold) &
    (df['purchase_amount'] > spend_threshold),
    'customer_value_segment'
] = 'VIP'

df.loc[
    (df['previous_purchases'] > purchase_threshold) &
    (df['purchase_amount'] <= spend_threshold),
    'customer_value_segment'
] = 'Loyal'

df.loc[
    (df['previous_purchases'] <= purchase_threshold) &
    (df['purchase_amount'] > spend_threshold),
    'customer_value_segment'
] = 'Potential'

In [55]:
df[['customer_id', 'previous_purchases', 'purchase_amount', 'customer_value_segment']].head()

,customer_id,previous_purchases,purchase_amount,customer_value_segment
0,1,14,53,Regular
1,2,2,64,Potential
2,3,23,73,Potential
3,4,49,90,VIP
4,5,31,49,Loyal


In [56]:
df['discount_flag'] = (
    df['discount_applied']
    .map({'Yes': 1, 'No': 0})
)

In [57]:
discount_summary = (
    df.groupby('customer_value_segment')
      .agg(
          customers=('customer_id', 'count'),
          avg_spend=('purchase_amount', 'mean'),
          discount_usage=('discount_flag', 'mean')
      )
      .reset_index()
)

discount_summary['discount_usage'] *= 100

discount_summary

,customer_value_segment,customers,avg_spend,discount_usage
0,Loyal,979,39.626149,44.024515
1,Potential,958,80.771399,43.632568
2,Regular,1007,39.463754,42.403178
3,VIP,956,80.719665,41.945607


In [58]:
product_analysis = (
    df.groupby('item_purchased')
      .agg(
          total_revenue=('purchase_amount', 'sum'),
          avg_purchase=('purchase_amount', 'mean'),
          avg_rating=('review_rating', 'mean'),
          orders=('customer_id', 'count')
      )
      .sort_values('total_revenue', ascending=False)
)

product_analysis.head(10)

,total_revenue,avg_purchase,avg_rating,orders
item_purchased,,,,
Blouse,10410,60.877193,3.680702,171
Shirt,10332,61.136095,3.622485,169
Dress,10320,62.168675,3.749398,166
Pants,10090,59.005848,3.720468,171
Jewelry,10010,58.538012,3.759064,171
Sunglasses,9649,59.931677,3.752174,161
Belt,9635,59.844720,3.763975,161
Scarf,9561,60.898089,3.707006,157
Sweater,9462,57.695122,3.759756,164


In [59]:
product_analysis[
    product_analysis['avg_rating'] >= 4
].sort_values('orders').head(10)

,total_revenue,avg_purchase,avg_rating,orders
item_purchased,,,,


In [60]:
category_analysis = (
    df.groupby('category')
      .agg(
          revenue=('purchase_amount', 'sum'),
          orders=('customer_id', 'count'),
          avg_order_value=('purchase_amount', 'mean'),
          avg_rating=('review_rating', 'mean')
      )
      .sort_values('revenue', ascending=False)
)

category_analysis

,revenue,orders,avg_order_value,avg_rating
category,,,,
Clothing,104264,1737,60.025331,3.721301
Accessories,74200,1240,59.838710,3.770242
Footwear,36093,599,60.255426,3.793823
Outerwear,18524,324,57.172840,3.745988


In [62]:

df.to_sql(
    'customer',
    engine,
    if_exists='replace',
    index=False
)

900

In [63]:
df['customer_value_segment'].value_counts()

,count
customer_value_segment,
Regular,1007
Loyal,979
Potential,958
VIP,956
